# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [37]:

# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display,update_display
from openai import OpenAI
import requests
# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [16]:
# constants
model_gemini='gemini-3-flash-preview'
model_gpt = 'gpt-4o-mini'
model_llama = 'llama3.2'


In [5]:
# creating gemini client

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")


gemini=OpenAI(api_key=google_api_key,base_url=GEMINI_BASE_URL)
print("your gemini client is ready to use")

API key found and looks good so far!
your gemini client is ready to use


In [39]:
#creating Ollama client
requests.get("http://localhost:11434").content
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
print("your ollama client is ready to use")

your ollama client is ready to use


In [9]:

system_prompt="""You are an expert technical instructor.

Explain concepts in a simple, clear, and structured way.

Your explanation MUST include:
1. Simple explanation (ELI5 style)
2. Technical explanation
3. Real-world example
4. (Optional) short summary

Avoid unnecessary complexity."""

In [ ]:
# request gemini to answer, with streaming
def get_response_for_question(question):
    #user propmt
    user_prompt=f""" here the question{question}"""
    #sending request
    stream = gemini.chat.completions.create(model=model_gemini, messages=[{"role": "user", "content":user_prompt},{"role":"system","content":system_prompt}],stream=True)
    response=""

    #displaying chunks
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
           response += chunk.choices[0].delta.content or ''
           update_display(Markdown(response), display_id=display_handle.display_id)   

In [56]:
question = """
Please explain what this code does and why:
yield display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id) }
"""

get_response_for_question(question)

This code is used to create a **streaming text effect** in a Jupyter Notebook or a similar interactive environment. It allows you to see an AI's response being typed out in real-time, rather than waiting for the entire message to finish before it appears.

---

### 1. Simple Explanation (ELI5)
Imagine you are watching someone write a letter. 

Without this code, the writer would go into a private room, write the whole letter, and then bring it out to show you all at once. You’d be sitting there staring at a blank wall while you wait.

With this code, it’s like the writer is standing at a **digital whiteboard** in front of you. 
1. First, they grab a spot on the whiteboard (the `display_handle`).
2. As soon as they think of a word, they write it down (`update_display`).
3. You get to watch the sentence grow word-by-word. It feels much faster and more interactive.

---

### 2. Technical Explanation

The code uses the **IPython display system** to modify the output of a cell after it has already been rendered. Here is the breakdown:

*   **`display(Markdown(""), display_id=True)`**: This creates a placeholder in the notebook's output area. By setting `display_id=True`, Jupyter gives this specific output a unique "ID." We store this in `display_handle`.
*   **`for chunk in stream`**: This loops through an iterator (likely from an OpenAI-style API). Instead of receiving one big string, the server sends small "chunks" of text as they are generated.
*   **`response += chunk...content or ''`**: This builds the full string bit by bit. It takes the new piece of text and adds it to the previous pieces.
*   **`update_display(Markdown(response), ...)`**: This is the magic part. Instead of printing a *new* line, it looks for the specific `display_id` we created earlier and **overwrites** its content with the updated, longer string.

**Why use this?**
Large Language Models (LLMs) can take 10–30 seconds to generate a long paragraph. If you don't stream, the user might think the program has crashed. Streaming provides immediate feedback and a better user experience (UX).

---

### 3. Real-World Example: AI Chatbots
Think of **ChatGPT** or **Claude**. When you ask them a question, the text "scrolls" or "types" onto the screen. 

If those apps didn't use this logic, you would see a loading spinner for 20 seconds, and then the entire essay would suddenly pop onto the screen. By using a "display handle" and "updating" the text as chunks arrive, the app feels "alive" and responsive.

---

### 4. Summary
This code creates a **persistent output area** and **updates it incrementally** with new data from a stream. It is the standard way to implement "typing" animations for AI models in Python notebooks to improve the user experience.

In [ ]:
# Get gemma4 answer

stream = ollama.chat.completions.create(model='gemma4', messages=[{"role": "user", "content":user_prompt},{"role":"system","content":system_prompt}],stream=True)
response=""


display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)   

This code snippet implements a classic pattern used in web applications and data science notebooks (like Jupyter) to display streaming results—meaning it shows output *as it is being generated*, rather than waiting for the entire response to finish.

Here is a detailed explanation of what this code does and why it uses this specific pattern.

---

### 💡 Simple Explanation (ELI5)

Imagine you ask a friend a very complicated question, and they don't answer all at once. Instead, they type out their answer, word by word, letting you see the words appear on the page.

This code is like the "typing effect" trigger for an answer box.

1.  **Setup:** First (the $\text{`yield display_handle = display(...)`}$ part), the code tells your computer: "Hey, reserve a specific little box on the screen to write in."
2.  **Loop:** Then, the code starts listening for small packets of incoming data ($\text{`for chunk in stream`}$), which are the words/parts of the answer.
3.  **Build & Update:** For every small packet it receives, it adds the new text to its running total ($\text{`response += ...`}$). Finally, it immediately tells the computer: "Update that reserved box and make it show *everything* that has appeared so far."

The key idea is that this process doesn't wait; it updates the screen instantly, creating the illusion of fast, live typing.

### 💻 Technical Explanation

This block of code is designed to manage the state and display of asynchronous, streaming data, typically from an API call (like a Large Language Model chat completion).

#### Breakdown by Line:

1.  **`yield display_handle = display(Markdown(""), display_id=True)`**
    *   **`display(...)`**: This function (likely from `IPython.display`) creates a visible HTML element in the notebook output.
    *   **`display_id=True`**: This is critical. It tells the environment to return a unique *ID* for that element. This ID is stored in `display_handle`.
    *   **`yield`**: Because this code is enclosed in a **generator function**, `yield` does not just run the code; it *pauses* and *returns* the control flow (and the reference to the display element) to the calling environment. This allows the display element to be initialized before the main processing loop begins.

2.  **`for chunk in stream:`**
    *   The code iterates over the `stream`. A `stream` is an iterable object that provides data chunks immediately as they become available, rather than providing all the data at once. This is asynchronous processing.

3.  **`response += chunk.choices[0].delta.content or ''`**
    *   **`chunk`**: This is one small package of data (a packet) of text received from the API.
    *   **`chunk.choices[0].delta.content`**: This is the deep, technical extraction of the actual text content from the structured API response object (e.g., the OpenAI API structure).
    *   **`response += ...`**: This concatenates (adds) the new text found in the current chunk to the running string variable, `response`.

4.  **`update_display(Markdown(response), display_id=display_handle.display_id)`**
    *   **`update_display(...)`**: This function (a wrapper around the display logic) targets the specific element we reserved earlier.
    *   **`display_id=display_handle.display_id`**: This tells the function *exactly* which element to update, preventing it from creating a new box every time.
    *   **Action**: It injects the growing content (`Markdown(response)`) into the existing box, making the text appear to constantly grow.

### 🌐 Real-World Example

**Scenario:** Building a chat interface powered by a sophisticated AI model.

**Without this code (Bad way):** The user types "Explain quantum physics." The program makes the API call, and the user stares at a blank screen for 10 seconds until the AI finishes writing "Quantum physics is..." all the way to "...conceptually complex."

**With this code (Good way):**
1. The user types the prompt.
2. The code executes the stream loop.
3. As the AI sends the first word, the code runs $\rightarrow$ Word appears.
4. As the AI sends the second word, the code runs $\rightarrow$ Word appears.
5. ...and so on.

The result is a fantastic **streaming experience** that feels fast, engaging, and responsive to the user.

### 📜 Short Summary

This code is a **streaming display mechanism** designed for dynamic, chunk-by-chunk output. It initializes a display container element using `yield` and then uses a loop to continuously read incoming data chunks from a stream. With each chunk, it appends the text to an internal variable and immediately redraws (updates) the original persistent display container, achieving a smooth, live typing effect for the end-user.